# MNIST Classification: ANN + FastAPI + Streamlit (Google Colab)

This single Colab notebook:
1. Trains an ANN on MNIST
2. Creates a FastAPI backend (`main.py`)
3. Creates a Streamlit frontend (`app.py`)
4. Launches both and gives you a **public link** to open the Streamlit app

**Run the cells from top to bottom** (Runtime > Run all also works).

## 1. Install requirements (run once)

In [ ]:
# Colab already has tensorflow, numpy, pandas, pillow, matplotlib and requests
%pip install -q fastapi "uvicorn[standard]" python-multipart streamlit

## 2. Load and explore the MNIST dataset
70,000 grayscale images of 28x28 pixels: 60,000 for training, 10,000 for testing, 10 classes (digits 0-9).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models

(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
print("Train:", x_train.shape, y_train.shape)
print("Test :", x_test.shape, y_test.shape)

# Show a few samples
plt.figure(figsize=(10, 2))
for i in range(10):
    plt.subplot(1, 10, i + 1)
    plt.imshow(x_train[i], cmap="gray")
    plt.title(y_train[i])
    plt.axis("off")
plt.show()

## 3. Preprocess and build the ANN

In [ ]:
# Normalize pixel values to 0-1
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

model = models.Sequential([
    layers.Input(shape=(28, 28)),
    layers.Flatten(),                        # 784 inputs
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax"),  # digits 0-9
])

model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
model.summary()

## 4. Train the model

In [ ]:
history = model.fit(x_train, y_train,
                    epochs=10, batch_size=128,
                    validation_split=0.1)

## 5. Evaluate and save

In [ ]:
loss, acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Test accuracy: {acc:.4f}")

# Training curves
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(history.history["accuracy"], label="train")
ax[0].plot(history.history["val_accuracy"], label="val")
ax[0].set_title("Accuracy"); ax[0].legend()
ax[1].plot(history.history["loss"], label="train")
ax[1].plot(history.history["val_loss"], label="val")
ax[1].set_title("Loss"); ax[1].legend()
plt.show()

model.save("mnist_ann.keras")
print("Model saved as mnist_ann.keras")

## 6. Create the FastAPI backend (`main.py`)
The `%%writefile` magic saves the cell content as a file.

In [ ]:
%%writefile main.py
import io

import numpy as np
import tensorflow as tf
from fastapi import FastAPI, File, HTTPException, UploadFile
from PIL import Image

app = FastAPI(title="MNIST Digit Classifier API", version="1.0")

model = tf.keras.models.load_model("mnist_ann.keras")


def preprocess(image_bytes: bytes) -> np.ndarray:
    # Convert an uploaded image into a (1, 28, 28) array like MNIST
    img = Image.open(io.BytesIO(image_bytes)).convert("L")
    img = img.resize((28, 28))
    arr = np.array(img, dtype="float32")

    # MNIST digits are white on black; invert if the image is mostly bright
    if arr.mean() > 127:
        arr = 255.0 - arr

    arr = arr / 255.0
    return arr.reshape(1, 28, 28)


@app.get("/")
def home():
    return {"message": "MNIST Digit Classifier API is running. Use POST /predict."}


@app.post("/predict")
async def predict(file: UploadFile = File(...)):
    if not file.content_type or not file.content_type.startswith("image/"):
        raise HTTPException(status_code=400, detail="Please upload an image file.")
    try:
        data = await file.read()
        x = preprocess(data)
    except Exception:
        raise HTTPException(status_code=400, detail="Could not read the image.")

    probs = model.predict(x, verbose=0)[0]
    digit = int(np.argmax(probs))
    return {
        "prediction": digit,
        "confidence": float(probs[digit]),
        "probabilities": {str(i): float(p) for i, p in enumerate(probs)},
    }

## 7. Create the Streamlit frontend (`app.py`)

In [ ]:
%%writefile app.py
import pandas as pd
import requests
import streamlit as st
from PIL import Image

API_URL = "http://127.0.0.1:8000/predict"

st.set_page_config(page_title="MNIST Digit Classifier", page_icon="🔢")
st.title("🔢 MNIST Digit Classifier")
st.write("Upload an image of a handwritten digit (0-9) and the ANN will predict it.")

uploaded = st.file_uploader("Choose an image", type=["png", "jpg", "jpeg"])

if uploaded is not None:
    image = Image.open(uploaded)
    col1, col2 = st.columns(2)
    col1.image(image, caption="Uploaded image", width=200)

    if st.button("Predict", type="primary"):
        try:
            files = {"file": (uploaded.name, uploaded.getvalue(), uploaded.type)}
            response = requests.post(API_URL, files=files, timeout=15)

            if response.status_code == 200:
                result = response.json()
                col2.metric("Predicted digit", result["prediction"])
                col2.write(f"Confidence: **{result['confidence'] * 100:.2f}%**")

                st.subheader("Class probabilities")
                df = pd.DataFrame(
                    {"Probability": list(result["probabilities"].values())},
                    index=list(result["probabilities"].keys()),
                )
                st.bar_chart(df)
            else:
                st.error(f"API error: {response.json().get('detail', 'Unknown error')}")
        except requests.exceptions.ConnectionError:
            st.error("Cannot reach the API. Run the 'Start the API' cell in the notebook.")

## 8. Start the API server (runs in the background)

In [ ]:
import subprocess, sys, time

api_proc = subprocess.Popen([sys.executable, "-m", "uvicorn", "main:app", "--port", "8000"])
time.sleep(8)  # give the server time to load the model
print("API running at http://127.0.0.1:8000  (docs: http://127.0.0.1:8000/docs)")

## 9. Test the API from the notebook

In [ ]:
import io, requests
from PIL import Image

# Take a random test image, send it to the API
idx = np.random.randint(len(x_test))
img = Image.fromarray((x_test[idx] * 255).astype("uint8"))
buf = io.BytesIO()
img.save(buf, format="PNG")

resp = requests.post("http://127.0.0.1:8000/predict",
                     files={"file": ("digit.png", buf.getvalue(), "image/png")})
result = resp.json()

plt.imshow(x_test[idx], cmap="gray"); plt.axis("off")
plt.title(f"True: {y_test[idx]} | Predicted: {result['prediction']} ({result['confidence']*100:.1f}%)")
plt.show()

## 10. Start the Streamlit frontend
Colab runs on a remote machine, so `localhost` won't open in your browser.
This cell creates a free public link (Cloudflare tunnel). Click the printed `trycloudflare.com` link.

In [ ]:
import re, os, subprocess, sys, time

# Download cloudflared (free tunnel, no account needed)
if not os.path.exists("cloudflared"):
    subprocess.run(["wget", "-q",
                    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
                    "-O", "cloudflared"], check=True)
    os.chmod("cloudflared", 0o755)

# Start Streamlit
st_proc = subprocess.Popen([sys.executable, "-m", "streamlit", "run", "app.py",
                            "--server.port", "8501", "--server.headless", "true"])
time.sleep(6)

# Open the public tunnel
cf_log = open("cf.log", "w")
cf_proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8501"],
                           stdout=cf_log, stderr=subprocess.STDOUT)

public_url = None
for _ in range(30):
    time.sleep(2)
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("cf.log").read())
    if m:
        public_url = m.group(0)
        break

if public_url:
    print("Open your app here:", public_url)
else:
    print("Tunnel URL not found yet. Run: print(open('cf.log').read())")

## 11. Stop the servers when finished

In [ ]:
cf_proc.terminate()
st_proc.terminate()
api_proc.terminate()
print("Servers stopped.")